# 05: Model registry and export (ADR-005)

1. Picks the tuned **logistic regression** run from EXP-003 (best by CV F1, never by test F1).
2. Registers it in the MLflow Model Registry as `telco-churn-classifier` and points the alias `champion` at it.
3. Loads it back **by alias** and checks it reproduces the logged test F1.
4. Exports it to a small folder (`model.skops` + `metadata.json`) and zips it for download. That folder is committed to the repo as `models/champion/` so the Docker image can serve it without Drive or MLflow.

Re-running registers a *new version* each time (aliases move, old versions stay). Run the register cell once.

In [ ]:
# --- AI Experiment Lab: standard Colab setup cell ---------------------------------
import os, subprocess, sys

REPO_URL = "https://github.com/sudhanshu230230/AI_Experiment_Lab.git"   # <-- edit
REPO_DIR = "/content/ai-experiment-lab"
DRIVE_ROOT = "/content/drive/MyDrive/ai-experiment-lab"
EXTRAS = "ml,tracking"          # pip extras from pyproject.toml

from google.colab import drive
drive.mount("/content/drive")
os.makedirs(f"{DRIVE_ROOT}/data", exist_ok=True)
os.makedirs(f"{DRIVE_ROOT}/mlflow", exist_ok=True)

# For a private repo, store a token in Colab Secrets as GITHUB_TOKEN (never paste it here).
url = REPO_URL
try:
    from google.colab import userdata
    token = userdata.get("GITHUB_TOKEN")
    url = REPO_URL.replace("https://", f"https://{token}@")
except Exception:
    pass

if os.path.isdir(REPO_DIR):
    subprocess.run(["git", "-C", REPO_DIR, "pull", "--ff-only"], check=True)
else:
    subprocess.run(["git", "clone", url, REPO_DIR], check=True)

subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-e", f"{REPO_DIR}[{EXTRAS}]"], check=True)

# An editable install adds a .pth file, which a running kernel does not re-read.
# Put src/ on sys.path so the import works in this session without a restart.
src_dir = f"{REPO_DIR}/src"
if src_dir not in sys.path:
    sys.path.insert(0, src_dir)

# Tracking store: SQLite database on Drive (accepted in ADR-001; the file: store is deprecated).
os.environ["MLFLOW_TRACKING_URI"] = f"sqlite:///{DRIVE_ROOT}/mlflow/mlflow.db"
os.environ["AEL_DATA_DIR"] = f"{DRIVE_ROOT}/data"

commit = subprocess.run(["git", "-C", REPO_DIR, "rev-parse", "--short", "HEAD"],
                        capture_output=True, text=True).stdout.strip()
print("Code version (commit):", commit)
print("Data dir:", os.environ["AEL_DATA_DIR"])

import ai_experiment_lab
print("ai_experiment_lab", ai_experiment_lab.__version__, "imported OK")

In [ ]:
from pathlib import Path
import shutil
import pandas as pd

from ai_experiment_lab.data import load_csv
from ai_experiment_lab.evaluation import classification_metrics
from ai_experiment_lab.preprocessing import split_data
from ai_experiment_lab.registry import export_pipeline, load_exported
from ai_experiment_lab.registry.mlflow_registry import (
    alias_version, find_run, load_by_alias, register_model, run_summary,
)
from ai_experiment_lab.tracking import backup_tracking_db, setup_tracking
from ai_experiment_lab.utils.config import get_data_dir, load_config

cfg = load_config(f"{REPO_DIR}/configs/default.yaml")
d, sp, seed = cfg["dataset"], cfg["split"], cfg["project"]["random_seed"]
EXPERIMENT = cfg["tracking"]["experiment_name"]
CHAMPION_MODEL = "logistic_regression"      # ADR-005
REGISTERED_NAME = "telco-churn-classifier"

setup_tracking(EXPERIMENT)
df, info = load_csv(Path(get_data_dir()) / d["file"])
split = split_data(df, d["target"], d["positive_label"], d["id_columns"],
                   sp["test_size"], seed, sp["stratify"])

run_id = find_run(EXPERIMENT, CHAMPION_MODEL, stage="tuned", order_by="cv_f1")
summary = run_summary(run_id)
print("source run:", run_id)
print("cv_f1 =", round(summary["metrics"]["cv_f1"], 4), "| test_f1 =", round(summary["metrics"]["test_f1"], 4))
print("code:", summary["code_version"], "| dataset:", (summary["dataset_sha256"] or "")[:12])

## Register and set the alias

In [ ]:
if alias_version(REGISTERED_NAME) is not None:
    print("Alias already set to version", alias_version(REGISTERED_NAME).version,
          "- registering again creates a NEW version and moves the alias.")
version = register_model(run_id, REGISTERED_NAME, alias="champion",
                         tags={"adr": "ADR-005", "experiment": "EXP-003"})
mv = alias_version(REGISTERED_NAME)
print(f"registered {REGISTERED_NAME} version {version}; alias 'champion' -> version {mv.version}")
print("lineage tag source_run_id:", mv.tags["source_run_id"])

## Load by alias and verify it reproduces the logged result

In [ ]:
pipe = load_by_alias(REGISTERED_NAME)            # models:/telco-churn-classifier@champion
pred = pipe.predict(split.X_test)
proba = pipe.predict_proba(split.X_test)[:, 1]
m = classification_metrics(split.y_test, pred, proba)
logged = summary["metrics"]
print(f"test_f1 from registry model: {m['f1']:.4f}   logged in run: {logged['test_f1']:.4f}")
print(f"test_roc_auc                : {m['roc_auc']:.4f}   logged in run: {logged['test_roc_auc']:.4f}")
ok = abs(m["f1"] - logged["test_f1"]) < 1e-6 and abs(m["roc_auc"] - logged["test_roc_auc"]) < 1e-6
print("REGISTRY MODEL REPRODUCES RUN" if ok else "MISMATCH: do not export; investigate")

## Export for serving

In [ ]:
out_dir = Path("/content/champion_export")
if out_dir.exists():
    shutil.rmtree(out_dir)
meta = {**summary, "registered_name": REGISTERED_NAME, "registry_version": int(mv.version),
        "alias": "champion", "positive_label": d["positive_label"], "threshold": 0.5,
        "adr": "ADR-005"}
export_pipeline(pipe, out_dir, meta)

# Prove the export works without MLflow: reload from the folder and compare predictions.
pipe2, meta2 = load_exported(out_dir)
import numpy as np
assert np.allclose(pipe2.predict_proba(split.X_test), pipe.predict_proba(split.X_test))
print("exported files:", sorted(p.name for p in out_dir.iterdir()))
print("input columns:", len(meta2["input_columns"]), "| trusted types:", meta2["skops_trusted_types"])

drive_copy = Path(DRIVE_ROOT) / "export" / "champion"
shutil.copytree(out_dir, drive_copy, dirs_exist_ok=True)
zip_path = shutil.make_archive("/content/champion", "zip", out_dir)
print("copy on Drive:", drive_copy, "| zip:", zip_path)

In [ ]:
from google.colab import files
files.download(zip_path)       # save it, then see the Windows commands in the chat

In [ ]:
print("backup:", backup_tracking_db())

## Record

Paste: the source-run lines, the registered version / alias lines, the `REGISTRY MODEL REPRODUCES RUN` line, the exported-files line, and the backup path. Keep the downloaded `champion.zip`: it becomes `models/champion/` in the repo.